In [5]:
import os
import gzip
import json
import pandas as pd
import random

# 1. Define Paths
# Using the exact path you provided in your terminal output
RAW_DATA_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\data\raw"
PROCESSED_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\data\processed"

# Ensure the processed directory exists
os.makedirs(PROCESSED_DIR, exist_ok=True)

# The 6 languages we need
LANGUAGES =["go", "java", "javascript", "php", "python", "ruby"]

# Define boilerplate words to filter out
BOILERPLATE =["copyright", "license", "author", "all rights reserved", "auto-generated"]

print("Setup Complete! Directories are ready.")

Setup Complete! Directories are ready.


In [6]:
def process_language(language, sample_size=2000):
    print(f"--- Processing Language: {language.upper()} ---")
    
    # Based on your description, the path is inside the language folder -> 'final' -> 'jsonl'
    search_path = os.path.join(RAW_DATA_DIR, language, "final", "jsonl")
    
    clean_records =[]
    
    # Walk through train, valid, and test folders
    for split in ["train", "valid", "test"]:
        split_dir = os.path.join(search_path, split)
        
        if not os.path.exists(split_dir):
            continue
            
        # Get all .jsonl.gz files in this split
        files =[f for f in os.listdir(split_dir) if f.endswith(".jsonl.gz")]
        
        for file in files:
            file_path = os.path.join(split_dir, file)
            
            # Open and read the gzipped jsonl file line by line
            with gzip.open(file_path, 'rt', encoding='utf-8') as f:
                for line in f:
                    data = json.loads(line)
                    
                    code_tokens = data.get("code_tokens",[])
                    docstring = data.get("docstring", "").strip()
                    
                    # FILTER 1: Skip if code is too long (> 512 tokens)
                    if len(code_tokens) > 512:
                        continue
                        
                    # FILTER 2: Skip if comment is empty
                    if not docstring:
                        continue
                        
                    # FILTER 3: Skip boilerplate comments
                    docstring_lower = docstring.lower()
                    if any(bp in docstring_lower for bp in BOILERPLATE):
                        continue
                        
                    # If it passes all filters, keep the essential parts!
                    # We only save what we need to save RAM memory
                    clean_records.append({
                        "language": language,
                        "repo": data.get("repo", ""),
                        "path": data.get("path", ""),
                        "func_name": data.get("func_name", ""),
                        "original_code": data.get("code", ""),
                        "original_comment": docstring,
                        "code_token_length": len(code_tokens)
                    })
                    
    print(f"Total clean records found for {language}: {len(clean_records)}")
    
    # Stratified Random Sampling
    if len(clean_records) >= sample_size:
        # Randomly select exactly 2000 records
        sampled_records = random.sample(clean_records, sample_size)
    else:
        print(f"WARNING: Not enough records for {language}. Found {len(clean_records)}, needed {sample_size}.")
        sampled_records = clean_records
        
    # Save the individual language sample to CSV just in case
    df = pd.DataFrame(sampled_records)
    output_file = os.path.join(PROCESSED_DIR, f"{language}_clean_sample.csv")
    df.to_csv(output_file, index=False)
    
    print(f"Saved {len(df)} samples to {output_file}\n")
    return df

In [7]:
all_dataframes =[]

# Loop through each language, process it, and append to our master list
for lang in LANGUAGES:
    df_lang = process_language(lang, sample_size=2000)
    all_dataframes.append(df_lang)

# Combine all 6 languages into one giant Master DataFrame
master_df = pd.concat(all_dataframes, ignore_index=True)

# Save the final merged original dataset
master_output_path = os.path.join(PROCESSED_DIR, "MASTER_original_12k.csv")
master_df.to_csv(master_output_path, index=False)

print("Master dataset successfully created and saved at:")
print(master_output_path)

--- Processing Language: GO ---
Total clean records found for go: 336833
Saved 2000 samples to C:\Users\HP\Desktop\thesis_preprocessing\data\processed\go_clean_sample.csv

--- Processing Language: JAVA ---
Total clean records found for java: 482408
Saved 2000 samples to C:\Users\HP\Desktop\thesis_preprocessing\data\processed\java_clean_sample.csv

--- Processing Language: JAVASCRIPT ---
Total clean records found for javascript: 131763
Saved 2000 samples to C:\Users\HP\Desktop\thesis_preprocessing\data\processed\javascript_clean_sample.csv

--- Processing Language: PHP ---
Total clean records found for php: 555541
Saved 2000 samples to C:\Users\HP\Desktop\thesis_preprocessing\data\processed\php_clean_sample.csv

--- Processing Language: PYTHON ---
Total clean records found for python: 443500
Saved 2000 samples to C:\Users\HP\Desktop\thesis_preprocessing\data\processed\python_clean_sample.csv

--- Processing Language: RUBY ---
Total clean records found for ruby: 52238
Saved 2000 samples 

In [8]:
# 1. Check if we have exactly 2,000 for each language (12,000 total)
print("--- Distribution by Language ---")
print(master_df['language'].value_counts())

print(f"\nTotal Dataset Size: {len(master_df)} rows.")

# 2. View the first 3 rows to make sure the data looks clean
print("\n--- Sample Data ---")
display(master_df.head(3))

# 3. Check the average length of the code tokens to ensure our <512 filter worked
print("\n--- Code Token Length Stats ---")
display(master_df['code_token_length'].describe())

--- Distribution by Language ---
language
go            2000
java          2000
javascript    2000
php           2000
python        2000
ruby          2000
Name: count, dtype: int64

Total Dataset Size: 12000 rows.

--- Sample Data ---


,language,repo,path,func_name,original_code,original_comment,code_token_length
0,go,aporeto-inc/trireme-lib,controller/pkg/urisearch/urisearch.go,FindRule,"func (c *APICache) FindRule(verb, uri string) ...",// FindRule finds a rule in the APICache witho...,76
1,go,gosuri/uilive,writer.go,Listen,func (w *Writer) Listen() {\n\tfor {\n\t\tsele...,// Listen listens for updates to the writer's ...,80
2,go,hyperledger/fabric-sdk-go,pkg/fab/mocks/mockdata.go,CreateBlockWithCCEvent,func CreateBlockWithCCEvent(events *pp.Chainco...,// CreateBlockWithCCEvent creates a mock block,39



--- Code Token Length Stats ---


count    12000.000000
mean        96.653250
std         82.837913
min         17.000000
25%         42.000000
50%         67.000000
75%        118.000000
max        511.000000
Name: code_token_length, dtype: float64